Kết nối Google Drive

In [11]:
import zipfile
from google.colab import drive

drive.mount('/content/drive/')

Mounted at /content/drive/


Các hàm hỗ trợ

In [20]:
def calculate_metrics(y_true, y_pred):
    # Đảm bảo chuyển về dạng mảng 1 chiều
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    epsilon = 1e-10

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean(np.square(y_true - y_pred)))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + epsilon))) * 100
    smape = np.mean(2.0 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + epsilon)) * 100

    return {
        'MAE': round(float(mae), 4),
        'RMSE': round(float(rmse), 4),
        'MAPE': f"{round(float(mape), 2)}%",
        'sMAPE': f"{round(float(smape), 2)}%"
    }

In [27]:
def create_features(df, target_col='load'):
    """Tạo các đặc trưng thời gian, Lag và Rolling"""
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month

    # Tạo Lag features
    df['lag_1'] = df[target_col].shift(1)
    df['lag_7'] = df[target_col].shift(7)

    # Tạo Rolling window features
    df['rolling_mean_7'] = df[target_col].shift(1).rolling(window=7).mean()
    return df

Model Naive

In [21]:
def naive_forecast(train_df, test_df, target_col='load'):
    last_value = train_df.iloc[-1][target_col]

    # Dự báo rolling: t = thực tế tại t-1
    predictions = test_df[[target_col]].copy()
    predictions['forecast'] = test_df[target_col].shift(1)

    # Điền giá trị đầu tiên bị NaN bằng giá trị cuối cùng của train
    predictions.iloc[0, predictions.columns.get_loc('forecast')] = last_value
    return predictions['forecast']

Moving Average

In [23]:
def create_features(df, target_col='load'):
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['month'] = df.index.month

    # Tạo Lag features
    df['lag_1'] = df[target_col].shift(1)
    df['lag_7'] = df[target_col].shift(7)

    # Tạo Rolling window features
    df['rolling_mean_7'] = df[target_col].shift(1).rolling(window=7).mean()
    return df

Load data

In [30]:
print("\n[INFO] Đang xử lý file data tổng...")
# Lưu ý: Sửa lại tên file 'data.csv' nếu file gốc của bạn tên khác
df_full = pd.read_csv('/content/drive/MyDrive/clean_hourly.csv', thousands=',')

df_full['time'] = pd.to_datetime(df_full['time'])
df_full = df_full.sort_values(by='time')
df_full.set_index('time', inplace=True)
df_full = df_full[['load']]

# Tạo features trên toàn bộ dữ liệu (đảm bảo lag/rolling không bị đứt quãng)
df_full_features = create_features(df_full, target_col='load')

# ==========================================
# 3. DÙNG 3 FILE TRAIN/VAL/TEST ĐỂ LẤY INDEX VÀ TÁCH TẬP
# ==========================================
print("[INFO] Đang map với 3 tập Train/Val/Test...")
def get_time_index(filepath, time_col='time'):
    df = pd.read_csv(filepath, thousands=',')
    return pd.to_datetime(df[time_col])

train_idx = get_time_index('/content/drive/MyDrive/train.csv')
val_idx   = get_time_index('/content/drive/MyDrive/val.csv')
test_idx  = get_time_index('/content/drive/MyDrive/test.csv')

# Trích xuất dữ liệu tương ứng từ bảng df_full_features
# Tập train bỏ đi các dòng NaN đầu tiên do tạo lag
train_features = df_full_features.loc[train_idx].dropna()
val_features   = df_full_features.loc[val_idx]
test_features  = df_full_features.loc[test_idx]

print(f"Kích thước Train: {train_features.shape}")
print(f"Kích thước Val:   {val_features.shape}")
print(f"Kích thước Test:  {test_features.shape}")


[INFO] Đang xử lý file data tổng...
[INFO] Đang map với 3 tập Train/Val/Test...
Kích thước Train: (24537, 6)
Kích thước Val:   (5260, 6)
Kích thước Test:  (5260, 6)


Chạy thử nghiệm các mô hình

In [31]:
target_col = 'load'

print("\n" + "="*40)
print("--- KẾT QUẢ MÔ HÌNH NAIVE (Trên tập Test) ---")
naive_preds = naive_forecast(train_features, test_features, target_col=target_col)
print(calculate_metrics(test_features[target_col], naive_preds))

print("\n" + "="*40)
print("--- KẾT QUẢ MOVING AVERAGE (Window=7, Trên tập Test) ---")
ma_preds = moving_average_forecast(train_features, test_features, window_size=7, target_col=target_col)
print(calculate_metrics(test_features[target_col], ma_preds))

print("\n" + "="*40)
print("--- KẾT QUẢ LINEAR REGRESSION (Trên tập Test) ---")
features = ['lag_1', 'lag_7', 'rolling_mean_7', 'dayofweek', 'month']

X_train, y_train = train_features[features], train_features[target_col]
X_test,  y_test  = test_features[features],  test_features[target_col]

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_preds = lr_model.predict(X_test)
print(calculate_metrics(y_test, lr_preds))


--- KẾT QUẢ MÔ HÌNH NAIVE (Trên tập Test) ---
{'MAE': 1046.0713, 'RMSE': 1383.9539, 'MAPE': '3.69%', 'sMAPE': '3.69%'}

--- KẾT QUẢ MOVING AVERAGE (Window=7, Trên tập Test) ---
{'MAE': 3180.5487, 'RMSE': 3900.7669, 'MAPE': '11.35%', 'sMAPE': '11.24%'}

--- KẾT QUẢ LINEAR REGRESSION (Trên tập Test) ---
{'MAE': 1977630478668.191, 'RMSE': 1977630478668.1912, 'MAPE': '6992495192.73%', 'sMAPE': '200.0%'}
